In [1]:
%cd ..
import numpy as np
import sys
import os
from os.path import join as pjoin


# root_rot_velocity (B, seq_len, 1)
# root_linear_velocity (B, seq_len, 2)
# root_y (B, seq_len, 1)
# ric_data (B, seq_len, (joint_num - 1)*3)
# rot_data (B, seq_len, (joint_num - 1)*6)
# local_velocity (B, seq_len, joint_num*3)
# foot contact (B, seq_len, 4)
def mean_variance(data_dir, save_dir, joints_num):
    file_list = os.listdir(data_dir)
    data_list = []
    num_nan = 0
    for file in file_list:
        data = np.load(pjoin(data_dir, file))
        if np.isnan(data).any():
            print(file)
            num_nan += 1
            continue
        data_list.append(data)

    data = np.concatenate(data_list, axis=0)
    print(data.shape)
    Mean = data.mean(axis=0)
    Std = data.std(axis=0)
    Std[0:1] = Std[0:1].mean() / 1.0
    Std[1:3] = Std[1:3].mean() / 1.0
    Std[3:4] = Std[3:4].mean() / 1.0
    Std[4: 4+(joints_num - 1) * 3] = Std[4: 4+(joints_num - 1) * 3].mean() / 1.0
    Std[4+(joints_num - 1) * 3: 4+(joints_num - 1) * 9] = Std[4+(joints_num - 1) * 3: 4+(joints_num - 1) * 9].mean() / 1.0
    Std[4+(joints_num - 1) * 9: 4+(joints_num - 1) * 9 + joints_num*3] = Std[4+(joints_num - 1) * 9: 4+(joints_num - 1) * 9 + joints_num*3].mean() / 1.0
    Std[4 + (joints_num - 1) * 9 + joints_num * 3: ] = Std[4 + (joints_num - 1) * 9 + joints_num * 3: ].mean() / 1.0

    assert 8 + (joints_num - 1) * 9 + joints_num * 3 == Std.shape[-1]

    np.save(pjoin(save_dir, 'Mean.npy'), Mean)
    np.save(pjoin(save_dir, 'Std.npy'), Std)
    print(num_nan)

    return Mean, Std

/home/guangxun/code/Waymo-3DSkelMo


In [2]:
import os
import numpy as np
import torch
from os.path import join as pjoin

def mean_variance_torch(data_dir, save_dir, joints_num, device='cuda'):

    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    
    file_list = os.listdir(data_dir)
    tensors = []

    
    for fname in file_list:
        arr = np.load(pjoin(data_dir, fname)).astype(np.float32)
        if np.isnan(arr).any():
            continue
        t = torch.from_numpy(arr).to(device)       # float32 by default
        tensors.append(t)

    data = torch.cat(tensors, dim=0)


    Mean = data.mean(dim=0)
    Std  = data.std(dim=0)  

    segs = [
        slice(0,1),
        slice(1,3),
        slice(3,4),
        slice(4, 4 + (joints_num - 1) * 3),
        slice(4 + (joints_num - 1) * 3, 4 + (joints_num - 1) * 9),
        slice(4 + (joints_num - 1) * 9, 4 + (joints_num - 1) * 9 + joints_num*3),
        slice(4 + (joints_num - 1) * 9 + joints_num * 3, None),
    ]
    for sl in segs:
        Std[sl] = Std[sl].mean()

    assert 8 + (joints_num - 1) * 9 + joints_num * 3 == Std.numel()

    Mean_np = Mean.cpu().numpy()
    Std_np  = Std.cpu().numpy()
    np.save(pjoin(save_dir, 'Mean.npy'), Mean_np)
    np.save(pjoin(save_dir, 'Std.npy'),  Std_np)

    return Mean_np, Std_np


In [ ]:
# The given data is used to double check if you are on the right track.
# reference1 = np.load('./five_sample_global/new_joints/Mean.npy')
# reference2 = np.load('./five_sample_global/new_joints/Std.npy')

In [3]:
if __name__ == '__main__':
    # data_dir = './waymo_150_relatives/new_joint_vecs/'
    # save_dir = './waymo_150_relatives/'
    data_dir = 'FID_data/new_joints/new_joint_vecs'
    save_dir = 'FID_data/new_joints'
    mean, std = mean_variance_torch(data_dir, save_dir, 22)
    
#     print(mean)
#     print(Std)

### Check if your data is correct. If it's aligned with the given reference, then it is right

In [ ]:
# abs(mean-reference1).sum()

In [ ]:
# abs(std-reference2).sum()